# FDA MAUDE ADEVRSE EVENT ANALYSIS: DEVICE RISK PATTERNS IN 2025

In [ ]:
# IMPORT PANDAS
import pandas as pd

# CHECK WORKING DIRECTORY
import os
# Update this path to your local directory before running
os.chdir(r"D:\data_analysis\FDA_MAUDE_Database_2025\analyse\for_github")
print(os.getcwd())

Three datasets were downloaded from the FDA:
- Master Record (through 2025)
- Device Data (2025)
- MAUDE Patient Records (received to date for 2025)

The datasets were examined for duplicates, invalid rows, data types, and date format inconsistencies. They were then cleaned and reduced to retain only relevant columns.

The common key across all three datasets was MDR_REPORT_KEY (FDA Medical Device Reporting key), which was used to merge them into a single consolidated dataset for analysis.

In [ ]:
# MERGE THE CLEANED AND REDUCED DATASET
# First merge the Master and Device dataset
device_unique = df_device.drop_duplicates(subset="MDR_REPORT_KEY")
merged1 = pd.merge(df_master, device_unique, on="MDR_REPORT_KEY", how="left")

# Next add Patient dataset
patient_unique = df_patient.drop_duplicates(subset="MDR_REPORT_KEY")

# Final merged dataset
final_merged = pd.merge(merged1, patient_unique, on="MDR_REPORT_KEY", how="left")

# Save the file to csv
final_merged.to_csv("fda_merged.csv", index=False)

# Check on the merged file
df_merged = pd.read_csv("fda_merged.csv")
print(df_merged.isnull().sum())

Note: Due to dataset size constraints, the full merged dataset is not included in this repository.  
The analysis is based on a cleaned and reduced dataset (`fda_project_final.csv`).

To reproduce the full pipeline, the original FDA datasets must be downloaded and merged as described in the notebook.

In [ ]:
# READ CSV
import csv
df = pd.read_csv("fda_merged_final.csv")
# df = pd.read_csv("fda_merged_final.csv", sep= ",", quoting=csv.QUOTE_NONE, on_bad_lines="skip", encoding="utf-8", engine="python")

In [ ]:
# TOP REPORTED DEVICE GROUPS + PERCENTAGE
top_devices = (df["DEVICE_GROUP"].value_counts().head(30).rename_axis("DEVICE_GROUP").reset_index(name="COUNT"))

# Calculate percentage using full dataset
total_reports = len(df)
top_devices["PERCENT"] = (top_devices["COUNT"] / total_reports) * 100
top_devices["PERCENT"] = top_devices["PERCENT"].round(2)

# Save to csv
top_devices.to_csv("top_30_device_group.csv", index=False)

The top two device groups—insulin and dental devices together account for nearly 30% of all reported adverse events, indicating a highly concentrated reporting pattern.

A significant proportion of reported events is associated with diabetes related devices, including insulin delivery systems and glucose monitoring technologies.

In [49]:
# DEVICES DOMINATING DEATH REPORTS

# Top devices contributing to deaths (excluding NO MATCH)
death_devices = (df[(df["EVENT_TYPE"] == "D") & (df["DEVICE_GROUP"] != "NO MATCH")]["DEVICE_GROUP"].value_counts().head(30).rename_axis("DEVICE_GROUP").reset_index(name="DEATH_COUNT"))
death_devices.to_csv("top_30_death_devices.csv", index=False)

# Death rate calculation (with threshold)
death_rate = (df[df["EVENT_TYPE"] == "D"]["DEVICE_GROUP"].value_counts().div(df["DEVICE_GROUP"].value_counts()).mul(100))

# Avoid misleading high death rates from low-sample devices
death_rate = (death_rate[df["DEVICE_GROUP"].value_counts() >= 500].dropna().sort_values(ascending=False).head(30).rename_axis("DEVICE_GROUP").reset_index(name="DEATH_RATE"))

# Save to csv
death_rate.to_csv("death_rate.csv", index=False)

Devices with the highest number of reported deaths are not always the ones with the highest death rates, indicating that volume and risk must be analysed separately.

In [ ]:
# EVENT COUNT IN DEVICES SERVICED BY THIRD PARTY 
third_party_event = df.groupby("SERVICED_BY_3RD_PARTY_FLAG")["EVENT_TYPE"].value_counts().reset_index()
third_party_event.columns = ["SERVICED_BY_3RD_PARTY_FLAG", "EVENT_TYPE", "COUNT"]
third_party_event.to_csv("third_party_event.csv", index=False)

In [ ]:
# EVENT COUNT RATE IN DEVICES SERVICED BY THIRD PARTY
third_party_event_rate = pd.crosstab(df["SERVICED_BY_3RD_PARTY_FLAG"], df["EVENT_TYPE"], normalize="index") * 100
third_party_event_rate.to_csv("third_party_event_rate.csv", index=True)

Devices serviced by third parties exhibited a predominantly malfunction driven reporting profile (~90%), while manufacturer serviced devices showed relatively higher proportions of injury and death events.

This difference is likely influenced by device type and clinical criticality, as third party servicing appears to be more common among lower risk, high volume devices such as insulin delivery systems.

Additionally, reports with unknown servicing information showed patterns consistent with incomplete or lower quality documentation, particularly in more severe event categories.

In [ ]:
# COUNT ALL THE MALFUNCTIONS BY DEVICE GROUP BECAUSE OF THIRD PARTY FLAG
third_party_service_malfunction = df[(df["SERVICED_BY_3RD_PARTY_FLAG"]=="Y") & (df["EVENT_TYPE"]=="M")]["DEVICE_GROUP"].value_counts().head(30).reset_index()
third_party_service_malfunction.columns = ["DEVICE_GROUP", "COUNT"]
third_party_service_malfunction.to_csv("third_party_service_malfunction.csv", index=False)

Certain device categories, particularly diabetic and mobility-related devices, show elevated malfunction rates following third-party servicing.

Manufacturer managed devices had much higher injury and death proportions

In [ ]:
# COMPARE OPERATOR VS EVENT TYPE
df_clean = df[(df["DEVICE_OPERATOR_CLEAN"] != "Incomplete / Malformed Code") & (df["EVENT_TYPE"] != "*")]
operator_event = pd.crosstab(df_clean["DEVICE_OPERATOR_CLEAN"], df_clean["EVENT_TYPE"], normalize="index") * 100
operator_event.to_csv("operator_event.csv", index=True)

Lay-person operated devices showed a predominantly malfunction-driven reporting pattern (84%), while health professional-operated devices exhibited a substantially higher proportion of injury events (33%).

Records with undefined event types (‘*’) were excluded to ensure meaningful interpretation of safety outcomes.

In [18]:
# DEVICE AVAILABILITY
device_available = pd.crosstab(df_clean["DEVICE_AVAILABILITY_CLEAN"], df_clean["EVENT_TYPE"], normalize="index") * 100
device_available.to_csv("device_available.csv", index=True)

Devices returned to manufacturers showed a much higher injury share (51.6%)

In [ ]:
# DEVICE REPORT PRODUCT CODE
device_report_product_code = (df.groupby("DEVICE_REPORT_PRODUCT_CODE").agg(COUNT=("DEVICE_REPORT_PRODUCT_CODE", "count"), DEVICE_NAME=("GENERIC_NAME_CLEAN", "first")).sort_values(by="COUNT", ascending=False).head(15).reset_index())
device_report_product_code.to_csv("device_report_product_code.csv", index=False)


Adverse event volume concentrated in diabetes technologies, severe events concentrated in cardiac and respiratory devices.

Not all medical device risks are equal. This analysis of 4 million FDA adverse event records reveals that while diabetes devices dominate reporting volume, cardiac and respiratory devices concentrate the most severe outcomes. Who operates a device and who services it significantly shapes the type of adverse event reported  suggesting that risk profiling needs to consider the full device ecosystem, not just the device itself.